# GCDRec e GCDIt (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/GCDRec_GCDIt_cpp.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `GCDRec` e `GCDIt`: gli algoritmi in C++, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/GCDRec_GCDIt_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/GCDRec_GCDIt_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `FibRec` (PascalCase).

## Ricorsione in coda: GCDRec e GCDIt

Una chiamata ricorsiva è *in coda* se è l'ultima operazione e il suo risultato è restituito così com'è: si può sostituire con l'aggiornamento dei parametri e un salto all'inizio, cioè con un ciclo (condizione del ciclo = negazione del caso base; chiamata in coda = assegnamento simultaneo dei parametri). Il risultato è lo stesso, il numero di passi ha lo stesso ordine di grandezza e la memoria extra scende da $\Theta(\text{profondità})$ a $\Theta(1)$.

**GCDRec**$(a,b)$ / **GCDIt**$(a,b)$ — *Input*: due naturali $a$, $b$. *Output*: il loro massimo comun divisore (algoritmo di Euclide). Con $(a,b)=(252,105)$ entrambi passano per le coppie $(252,105)$, $(105,42)$, $(42,21)$, $(21,0)$ e restituiscono $21$: quattro chiamate contro tre iterazioni. La variabile temporanea $r$ serve perché $a\bmod b$ va calcolato con il *vecchio* $a$.

In [1]:
%%writefile gcd_rec_it.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

vector<string> gcdLog;   // coppie (a, b) visitate (solo per la stampa)

string PairStr(long long a, long long b)
{
    return "(" + to_string(a) + "," + to_string(b) + ")";
}

// GCDRec(a, b)
long long GCDRec(long long a, long long b)
{
    gcdLog.push_back(PairStr(a, b));
    if (b == 0)
    {
        return a;
    }
    return GCDRec(b, a % b);            // chiamata in coda
}

// GCDIt(a, b)
long long GCDIt(long long a, long long b)
{
    gcdLog.push_back(PairStr(a, b));
    while (b != 0)                      // negazione del caso base
    {
        long long r = a % b;            // con il vecchio a
        a = b;
        b = r;
        gcdLog.push_back(PairStr(a, b));
    }
    return a;
}

#ifndef NOMAIN
int main()
{
    long long pairsAB[][2] = {{252, 105}, {1071, 462}, {832040, 514229}};
    for (auto &p : pairsAB)
    {
        long long a = p[0], b = p[1];
        gcdLog.clear();
        long long g1 = GCDRec(a, b);
        int calls = gcdLog.size();
        string pairs;
        for (size_t i = 0; i < gcdLog.size(); i++)
        {
            if (i > 0) pairs += " ";
            pairs += gcdLog[i];
        }
        gcdLog.clear();
        long long g2 = GCDIt(a, b);
        int iters = (int)gcdLog.size() - 1;
        printf("GCDRec(%lld, %lld) = %lld   chiamate=%d     GCDIt = %lld   iterazioni=%d   %s\n", a, b, g1, calls, g2, iters, g1 == g2 ? "OK" : "ERROR");
        if (a == 252)
        {
            printf("  coppie: %s\n", pairs.c_str());
        }
    }
    return 0;
}
#endif

Writing gcd_rec_it.cpp


In [2]:
!g++ -std=c++17 -O2 -o gcd_rec_it gcd_rec_it.cpp && ./gcd_rec_it

GCDRec(252, 105) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
  coppie: (252,105) (105,42) (42,21) (21,0)
GCDRec(1071, 462) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
GCDRec(832040, 514229) = 1   chiamate=29     GCDIt = 1   iterazioni=28   OK


**Prova tu.** Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.

In [3]:
%%writefile prova_gcd_rec_it.cpp
#define NOMAIN
#include "gcd_rec_it.cpp"

int main()
{
    // Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.
    gcdLog.clear();
    long long g = GCDRec(46368, 28657);
    printf("%lld %d\n", g, (int)gcdLog.size());
    return 0;
}

Writing prova_gcd_rec_it.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_gcd_rec_it prova_gcd_rec_it.cpp && ./prova_gcd_rec_it

1 23


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).